# Loss-Aware Synthetic Data Generation - Strong Colab Run

Use this notebook in Google Colab with a GPU runtime.

Recommended setup: `Runtime > Change runtime type > GPU`.

This notebook runs the stronger configuration in `config/strong.json`.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Get The Project Code

Option A: if your project is on GitHub, set `REPO_URL` and run the clone cell.

Option B: if you uploaded the project folder manually to Colab, skip the clone cell and set `PROJECT_DIR` to that folder.

In [ ]:
# Option A: GitHub clone
# Replace this with your repository URL, then run this cell.
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
PROJECT_DIR = "/content/ethics"

import os
from pathlib import Path

if "YOUR_USERNAME" not in REPO_URL:
    if not Path(PROJECT_DIR).exists():
        !git clone {REPO_URL} {PROJECT_DIR}
    else:
        %cd {PROJECT_DIR}
        !git pull
else:
    print("Set REPO_URL first, or upload the project folder and set PROJECT_DIR manually.")

In [ ]:
# Option B: manual folder path
# If you uploaded the project manually, edit this path.
# PROJECT_DIR = "/content/ethics"

import os
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())
!ls

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Verify Imports And Strong Config

In [ ]:
import os, json
os.environ["PYTHONPATH"] = "src"

with open("config/strong.json", "r", encoding="utf-8") as f:
    strong_config = json.load(f)

print(json.dumps(strong_config["dataset"], indent=2))
print(json.dumps(strong_config["synthetic_generation"], indent=2))

## 5. Run The Strong Pipeline

This may take a long time because CTGAN and TVAE use 100 epochs in `config/strong.json`.

Do not add `--reuse-existing` for the strong run unless you intentionally want to reuse existing strong outputs.

In [ ]:
!PYTHONPATH=src python -m lossaware.run_pipeline --config config/strong.json

## 6. Inspect Final Results

In [ ]:
import pandas as pd
from pathlib import Path

result_dir = Path("results/folktables_acs_income")

expected = [
    "baseline_results.csv",
    "gaussian_copula.csv",
    "ctgan.csv",
    "tvae.csv",
    "utility_summary.csv",
    "privacy_utility_tradeoff.csv",
    "loss_aware_ranking.csv",
    "final_results_summary.csv",
]

print("Existing result files:")
for path in sorted(result_dir.glob("*.csv")):
    print("-", path.name)

missing = [name for name in expected if not (result_dir / name).exists() and not Path("data/synthetic/folktables_acs_income", name).exists()]
if missing:
    print("\nMissing outputs:", missing)
    print("Run this resume command, then rerun this cell:")
    print("!PYTHONPATH=src python -m lossaware.run_pipeline --config config/strong.json --reuse-existing")
else:
    display(pd.read_csv(result_dir / "baseline_results.csv"))
    display(pd.read_csv(result_dir / "final_results_summary.csv"))
    display(pd.read_csv(result_dir / "loss_aware_ranking.csv"))

In [ ]:
from pathlib import Path

summary_path = Path("reports/final_summary.md")
if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("reports/final_summary.md does not exist yet.")
    print("Resume the pipeline first:")
    print("!PYTHONPATH=src python -m lossaware.run_pipeline --config config/strong.json --reuse-existing")

## 7. Save Outputs To Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/lossaware_results
!cp -r results /content/drive/MyDrive/lossaware_results/
!cp -r reports /content/drive/MyDrive/lossaware_results/

print("Saved to /content/drive/MyDrive/lossaware_results")

## 8. Optional: Zip Results For Download

In [ ]:
!zip -qr lossaware_strong_results.zip results reports
from google.colab import files
files.download("lossaware_strong_results.zip")